# Baseline Churn Models

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from churn_mlops.data import load_raw_data, validate_data
from churn_mlops.evaluation import Timer, evaluate_model
from churn_mlops.models import MODEL_REGISTRY, FeatureBuilder, build_classifier_pipeline

pd.set_option('display.width', 200)

## Load Training Data

In [ ]:
# load raw data
df_train = load_raw_data("customer_churn_dataset-training.csv", index_col="customerid")

# validate data contract/schema
df_train = validate_data(df_train)

# separate column names into numeric and categorical
categorical_features = df_train.columns[df_train.dtypes == 'string'].to_list()
numeric_features_raw = df_train.columns[df_train.dtypes != 'string'].to_list()
numeric_features_raw.remove('churn')
numeric_features = FeatureBuilder().get_feature_names_out(numeric_features_raw)

## Train/Test Split

In [ ]:
X, y = df_train.drop(columns=['churn']), df_train['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

## Baseline Models

In [ ]:
# define baseline classifier pipelines
baseline = {}
for n in MODEL_REGISTRY:
    clf = MODEL_REGISTRY[n]["clf"]
    default_params = MODEL_REGISTRY[n]["default_params"]
    classifier = clf(**default_params)
    baseline[n] = build_classifier_pipeline(classifier)

In [ ]:
# fit baseline models on train-split
fit_timer = {}
for n, pipe in baseline.items():
    with Timer() as timer:
        pipe.fit(X_train, y_train)
    fit_timer[n] = float(timer.duration)

## Model Evaluation / Comparison
* Metrics: ROC AUC, PR AUC, Confusion Matrix, Classification Report
* Tabular comparision
* Error Analysis: FP vs FN
* ~~Segment Performance (ROC AUC by gender, subscription type, contract length)~~
* Feature Importance / ~~SHAP~~

In [ ]:
# Predict probabilities
pred_timer = {}
pred_proba = {}
for n, pipe in baseline.items():
    with Timer() as timer:
        pred_proba[n] = pipe.predict_proba(X_test)[:, 1]
    pred_timer[n] = float(timer.duration)

In [ ]:
# predictions based on EDA results
def eda_predict(X):
    return (
        (X["age"] > 50)
        | (X["support_calls"] >= 6)
        | (X["payment_delay"] > 20)
        | (X["total_spend"] <= 500)
        | (X["contract_length"] == "Monthly")
    ).astype(int)

with Timer() as timer:
    pred_proba["eda"] = eda_predict(X_test)
pred_timer["eda"] = float(timer.duration)

In [ ]:
# Compute evaluation metrics
results = []
for n, pipe in baseline.items():
    results.append(
        evaluate_model(
            pipe.named_steps['classifier'].__class__.__name__,
            y_test,
            pred_proba[n],
            fit_timer[n],
            pred_timer[n],
        )
    )
results.append(evaluate_model("EDA based", y_test, pred_proba["eda"], 0, pred_timer["eda"]))
results_df = pd.DataFrame([vars(result) for result in results])
print(results_df)

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred_proba["eda"]))

In [ ]:
feature_importance_rf = (
    pd.DataFrame(
        {
            "feature": baseline['rf'].named_steps['preprocessor'].get_feature_names_out(),
            "importance": baseline['rf'].named_steps['classifier'].feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
)
print(feature_importance_rf)

## Findings and Limitations

### Near-Perfect Model Performance

| Model | ROC AUC |
|--------|---------:|
| EDA-based rule benchmark | 0.972092 |
| Logistic Regression | 0.960841 |
| Random Forest | 0.999998 |
| HistGradientBoosting | 1.000000 |

### Investigation

The following checks were performed to validate the unusually high model performance:

- Reviewed feature importance rankings
- Inspected prediction probability distributions
- Analyzed confusion matrices
- Revisited findings from the exploratory data analysis (EDA)

### Findings

EDA revealed several variables exhibiting deterministic or near-deterministic relationships with the target variable. Examples include:

- Age above 50 years
- Six or more support calls
- Payment delays greater than 20 days
- Total spend below 500
- Monthly contract length

Tree-based models were able to exploit these relationships and achieved nearly perfect predictive performance on the holdout test set.

### Conclusion

The dataset appears to contain highly structured relationships between predictor variables and the target variable that are unlikely to occur in real-world churn prediction problems.

Possible explanations include:

- The dataset is synthetic and generated from predefined business rules.
- The target variable is strongly derived from a subset of the predictor variables.
- Information leakage may be present in the dataset design.

The available evidence does not allow these explanations to be distinguished conclusively, but our initial EDA strongly points towards synthetic data.

### Impact

Model evaluation metrics reported in this project should not be interpreted as representative of production-grade churn prediction performance. Instead, the primary purpose of this exercise is to demonstrate the end-to-end machine learning workflow, including data preparation, model development, evaluation, explainability, and MLOps practices.

## Dataset Split Strategy

The original Kaggle dataset provides separate "training" and "testing" datasets.

For this project, the provided testing dataset was renamed to **inference dataset** and is not used during model development.

The rationale is to emulate a production machine learning workflow:

- The training dataset is used for model development, including:
  - train/test splitting
  - cross-validation
  - model/feature selection
  - hyperparameter tuning
  - evaluation

- The inference dataset is reserved as a proxy for unseen production data and intended to be used for:
  - batch inference
  - model monitoring
  - data drift detection
  - concept drift analysis

This approach more closely resembles a real-world MLOps setting, where future production data is not available during model development.

# Evaluation Based On Separate "Test Data"

Let's briefly explore how our models perform on the inference dataset:

In [ ]:
# load raw data
df_inference = load_raw_data("customer_churn_dataset-inference.csv", index_col="customerid")

# validate data contract/schema
df_inference = validate_data(df_inference)

# split off target
X_inference, y_inference = df_inference.drop(columns=['churn']), df_inference['churn']
print(X_inference.shape, y_inference.shape)

In [ ]:
# Predict probabilities
pred_timer_inference = {}
pred_proba_inference = {}
for n, pipe in baseline.items():
    with Timer() as timer:
        pred_proba_inference[n] = pipe.predict_proba(X_inference)[:, 1]
    pred_timer_inference[n] = float(timer.duration)

# predictions based on EDA results
with Timer() as timer:
    pred_proba_inference["eda"] = eda_predict(X_inference)
pred_timer_inference["eda"] = float(timer.duration)

In [ ]:
# Compute evaluation metrics
results_inference = []
for n, pipe in baseline.items():
    results_inference.append(
        evaluate_model(
            pipe.named_steps['classifier'].__class__.__name__,
            y_inference,
            pred_proba_inference[n],
            None,
            pred_timer_inference[n],
        )
    )
results_inference.append(evaluate_model("EDA based", y_inference, pred_proba_inference["eda"], None, pred_timer_inference["eda"]))
results_inference_df = pd.DataFrame([vars(result) for result in results_inference])
print(results_inference_df)

## Conclusion

Although tree-based models achieved near-perfect performance on the internal validation split, performance dropped substantially when evaluated on the separate testing dataset. This discrepancy suggests that the internal validation scores may overestimate real-world performance and highlights the importance of evaluating models on truly unseen data. The inference dataset will therefore be used to investigate potential distribution shift, model robustness, and monitoring strategies.